<a href="https://colab.research.google.com/github/jaysulk/GENERIC-FNO/blob/main/GENERIC_FNO_Figures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
# ============================================================================
# generic_fno_figures_v3.py  --  paper figures for GENERIC-FNO
#
# Produces (to FIGDIR, as both .pdf and .png):
#   MAIN TEXT
#     fig_accuracy.pdf       rollout L2 +/- std, per backbone (wave excluded)
#     fig_dissipation.pdf    gauge-invariant r_mech per PDE x backbone
#     fig_resolution.pdf     zero-shot super-resolution: L2 + r_E vs grid
#     fig_interpretability.pdf  E flat / S rising / mechE decaying along rollout
#                               (LIVE: needs torch + model module + checkpoints)
#   APPENDIX
#     fig_degeneracy.pdf     machine-precision degeneracy residuals (log scale)
#     fig_gauge_variance.pdf per-seed rho_M (scatters) vs r_mech (stable)
#     fig_rmech_ordering.pdf r_mech vs ground-truth pi_true (ordering match)
#     fig_euler_rk4.pdf      advection: Euler vs RK4 L2 and r_E vs grid
#
# DATA SOURCE
#   The scalar figures use the VERIFIED seed-averaged numbers from the
#   2026-06-05 three-seed runs (2D FNO, 1D FNO, 1D DeepONet), embedded below.
#   They match the run logs to the digit, so the figures are reproducible with
#   no Drive / pickle dependency.  To regenerate from your own pickles instead,
#   wire up load_from_pkls() (stub at the bottom) and call it in main().
#
#   fig_interpretability is the one figure that needs trajectory arrays; it
#   loads a GENERIC checkpoint and rolls the model out.  It is skipped with a
#   message if torch / the model module / checkpoints are unavailable.
# ============================================================================
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")                       # headless: the usual reason plots "don't work"
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
BASE   = os.environ.get("GENERIC_FNO_BASE",
                        "/content/drive/MyDrive/GENERIC_FNO_results")
FIGDIR = os.environ.get("GENERIC_FNO_FIGDIR", os.path.join(BASE, "figures"))

PDES   = ["heat", "advection", "burgers"]
PDE_LABEL = {"heat": "Heat", "advection": "Advection", "burgers": "Burgers"}
C = {"FNO": "#7f7f7f", "EP-FNO": "#1f77b4", "GENERIC": "#d62728",
     "DeepONet": "#7f7f7f", "truth": "#000000",
     "2D FNO": "#d62728", "1D FNO": "#ff7f0e", "DeepONet_bk": "#2ca02c"}

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 200, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "legend.frameon": False, "axes.axisbelow": True,
})

# ----------------------------------------------------------------------------
# VERIFIED seed-averaged data (mean, std) -- rollout L2
# ----------------------------------------------------------------------------
ACC = {
 "2D FNO": {
   "heat":      {"FNO": (0.113, 0.034), "EP-FNO": (0.111, 0.008), "GENERIC": (0.091, 0.003)},
   "advection": {"FNO": (0.119, 0.004), "EP-FNO": (0.132, 0.006), "GENERIC": (0.179, 0.020)},
   "burgers":   {"FNO": (0.175, 0.025), "EP-FNO": (0.138, 0.011), "GENERIC": (0.026, 0.003)}},
 "1D FNO": {
   "heat":      {"FNO": (0.082, 0.015), "EP-FNO": (0.065, 0.008), "GENERIC": (0.095, 0.012)},
   "advection": {"FNO": (0.118, 0.004), "EP-FNO": (0.114, 0.014), "GENERIC": (0.233, 0.022)},
   "burgers":   {"FNO": (0.073, 0.010), "EP-FNO": (0.056, 0.003), "GENERIC": (0.046, 0.006)}},
 "1D DeepONet": {
   "heat":      {"DeepONet": (0.057, 0.004), "GENERIC": (0.008, 0.001)},
   "advection": {"DeepONet": (0.201, 0.009), "GENERIC": (0.018, 0.002)},
   "burgers":   {"DeepONet": (0.020, 0.002), "GENERIC": (0.016, 0.001)}},
}
MODELS = {"2D FNO": ["FNO", "EP-FNO", "GENERIC"],
          "1D FNO": ["FNO", "EP-FNO", "GENERIC"],
          "1D DeepONet": ["DeepONet", "GENERIC"]}

# gauge-invariant dissipation: (r_mech_mean, r_mech_std, pi_true)
RMECH = {
 "2D FNO":      {"heat": (0.306, 0.114, 2.93e-2), "advection": (0.003, 0.002, 1.39e-7), "burgers": (0.011, 0.004, 1.22e-3)},
 "1D FNO":      {"heat": (0.267, 0.102, 1.70e-2), "advection": (0.029, 0.028, 1.03e-7), "burgers": (0.051, 0.024, 1.57e-3)},
 "1D DeepONet": {"heat": (0.825, 0.019, 1.70e-2), "advection": (0.007, 0.008, 7.42e-8), "burgers": (0.252, 0.048, 1.37e-3)},
}

# zero-shot super-resolution (Euler model of record), trained @64
RES = [64, 96, 128, 192, 256]
SUPERRES = {
 "heat":    {"FNO": [0.112, 0.097, 0.099, 0.091, 0.099],
             "GENERIC": [0.040, 0.026, 0.023, 0.022, 0.023],
             "rE": [3.1e-7, 7.2e-7, 9.7e-7, 1.1e-6, 3.2e-6]},
 "burgers": {"FNO": [0.107, 0.145, 0.131, 0.164, 0.124],
             "GENERIC": [0.033, 0.023, 0.021, 0.019, 0.015],
             "rE": [4.1e-6, 7.7e-6, 1.2e-5, 1.1e-5, 3.7e-5]},
}

# machine-precision verification residuals (16x16 random init), should be ~0
DEGEN = [
 (r"$\langle\delta E, L\,\delta E\rangle$  (skewness)",       7e-19),
 (r"$\langle\delta S, L\,\delta E\rangle$  (entropy degen.)", 1e-15),
 (r"$\langle\delta E, M\,\delta S\rangle$  (energy degen.)",  3e-13),
 (r"$\langle\delta E, \partial_t u\rangle$  (energy cons.)",  3e-13),
 (r"$r_E$ (trained models, rollout)",                          1e-6),
]
MACHINE_EPS = 2.22e-16

# per-seed rho_M (gauge-dependent) and r_mech (gauge-invariant)
RHOM_SEEDS = {
 "2D FNO":      {"heat": [0.9993, 0.9994, 0.9997], "advection": [0.0004, 0.0001, 0.0004], "burgers": [0.0015, 0.9995, 0.9999]},
 "1D FNO":      {"heat": [0.7728, 0.5427, 0.2984], "advection": [0.0033, 0.0062, 0.0133], "burgers": [0.1975, 0.9253, 0.2547]},
 "1D DeepONet": {"heat": [0.9067, 0.9313, 0.8031], "advection": [0.0238, 0.0378, 0.0305], "burgers": [0.3427, 0.5703, 0.6879]},
}
RMECH_SEEDS = {
 "2D FNO":      {"heat": [0.1450, 0.3729, 0.3995], "advection": [0.0011, 0.0063, 0.0015], "burgers": [0.0086, 0.0085, 0.0165]},
 "1D FNO":      {"heat": [0.3532, 0.3246, 0.1236], "advection": [0.0104, 0.0080, 0.0676], "burgers": [0.0232, 0.0818, 0.0466]},
 "1D DeepONet": {"heat": [0.8189, 0.8507, 0.8055], "advection": [0.0176, -0.0010, 0.0030], "burgers": [0.1859, 0.2985, 0.2708]},
}

# advection Euler vs RK4+AA (limitations), trained @64
ERK_RES = [64, 128, 256]
EULER_RK4 = {
 "FNO":        {"L2": [0.174, 0.134, 0.117]},
 "Euler":      {"L2": [0.606, 0.154, 0.161], "rE": [2e-8, 9e-8, 1e-7]},
 "RK4":        {"L2": [0.341, 0.153, 0.159], "rE": [2.9e-3, 1.2e-3, 8e-5]},
}


# ----------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------
def _save(fig, name):
    os.makedirs(FIGDIR, exist_ok=True)
    for ext in ("pdf", "png"):
        p = os.path.join(FIGDIR, f"{name}.{ext}")
        fig.savefig(p, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {os.path.join(FIGDIR, name)}.pdf/.png")


# ----------------------------------------------------------------------------
# MAIN-TEXT FIGURES
# ----------------------------------------------------------------------------
def fig_accuracy():
    backbones = ["2D FNO", "1D FNO", "1D DeepONet"]
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    for ax, bk in zip(axes, backbones):
        models = MODELS[bk]
        x = np.arange(len(PDES)); w = 0.8 / len(models)
        for j, m in enumerate(models):
            means = [ACC[bk][p][m][0] for p in PDES]
            stds  = [ACC[bk][p][m][1] for p in PDES]
            ax.bar(x + (j - (len(models) - 1) / 2) * w, means, w, yerr=stds,
                   capsize=3, label=m, color=C[m], edgecolor="black", linewidth=0.4)
        ax.set_yscale("log")
        ax.set_xticks(x); ax.set_xticklabels([PDE_LABEL[p] for p in PDES])
        ax.set_title(bk); ax.set_ylabel(r"rollout $L^2$ (log)")
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(axis="y", ls=":", alpha=0.5)
    fig.suptitle("Predictive accuracy (10-step rollout, 3-seed mean$\\pm$std; wave excluded)", y=1.03)
    _save(fig, "fig_accuracy")


def fig_dissipation():
    backbones = list(RMECH.keys())
    fig, ax = plt.subplots(figsize=(7.4, 3.6))
    x = np.arange(len(PDES)); w = 0.8 / len(backbones)
    bkcol = {"2D FNO": "#d62728", "1D FNO": "#ff7f0e", "1D DeepONet": "#2ca02c"}
    for j, bk in enumerate(backbones):
        means = [RMECH[bk][p][0] for p in PDES]
        stds  = [RMECH[bk][p][1] for p in PDES]
        ax.bar(x + (j - (len(backbones) - 1) / 2) * w, means, w, yerr=stds,
               capsize=3, label=bk, color=bkcol[bk], edgecolor="black", linewidth=0.4)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels([PDE_LABEL[p] for p in PDES])
    ax.set_ylabel(r"gauge-invariant dissipation $r_{\mathrm{mech}}$")
    ax.set_title(r"Reversible advection $\to r_{\mathrm{mech}}\approx 0$;  ordering matches ground truth")
    ax.annotate(r"advection $\approx 0$" + "\n(reversible)", xy=(1, 0.02), xytext=(1.15, 0.45),
                fontsize=8, ha="left", arrowprops=dict(arrowstyle="->", lw=0.7))
    ax.legend(fontsize=8); ax.grid(axis="y", ls=":", alpha=0.5)
    _save(fig, "fig_dissipation")


def fig_resolution():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    # left: L2 vs resolution
    ax = axes[0]
    for p, ls in (("heat", "-"), ("burgers", "--")):
        ax.plot(RES, SUPERRES[p]["FNO"], ls, color=C["FNO"], marker="o", ms=4,
                label=f"FNO ({p})")
        ax.plot(RES, SUPERRES[p]["GENERIC"], ls, color=C["GENERIC"], marker="s", ms=4,
                label=f"GENERIC ({p})")
    ax.axvline(64, color="black", ls=":", lw=0.8)
    ax.text(64, ax.get_ylim()[1] * 0.92, " train grid", fontsize=8)
    ax.set_xlabel("evaluation resolution"); ax.set_ylabel(r"rollout $L^2$")
    ax.set_title("Zero-shot super-resolution accuracy")
    ax.set_xticks(RES); ax.legend(fontsize=7.5); ax.grid(ls=":", alpha=0.5)
    # right: r_E vs resolution
    ax = axes[1]
    ax.plot(RES, SUPERRES["heat"]["rE"], "-o", ms=4, color="#9467bd", label="heat")
    ax.plot(RES, SUPERRES["burgers"]["rE"], "--s", ms=4, color="#8c564b", label="burgers")
    ax.axvline(64, color="black", ls=":", lw=0.8)
    ax.set_yscale("log"); ax.set_xlabel("evaluation resolution")
    ax.set_ylabel(r"structural residual $r_E$ (log)")
    ax.set_title("Guarantees transfer too ($r_E$ stays negligible)")
    ax.set_xticks(RES); ax.legend(fontsize=8); ax.grid(ls=":", alpha=0.5)
    _save(fig, "fig_resolution")


# ----------------------------------------------------------------------------
# APPENDIX FIGURES
# ----------------------------------------------------------------------------
def fig_degeneracy():
    labels = [d[0] for d in DEGEN][::-1]
    vals   = [d[1] for d in DEGEN][::-1]
    fig, ax = plt.subplots(figsize=(7.6, 3.0))
    y = np.arange(len(labels))
    ax.barh(y, vals, color="#d62728", edgecolor="black", linewidth=0.4, height=0.6)
    ax.axvline(MACHINE_EPS, color="black", ls="--", lw=1.0)
    ax.text(MACHINE_EPS, len(labels) - 0.4, " float64 $\\epsilon$", fontsize=8, rotation=90, va="top")
    ax.set_xscale("log"); ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlim(1e-20, 1e-4); ax.set_xlabel("residual magnitude (log)")
    ax.set_title("Structural identities hold to machine precision (any init/dim/res)")
    ax.grid(axis="x", ls=":", alpha=0.5)
    _save(fig, "fig_degeneracy")


def fig_gauge_variance():
    backbones = list(RHOM_SEEDS.keys())
    dpdes = ["heat", "burgers"]                       # the non-reversible ones
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharex=True)
    groups = [(bk, p) for bk in backbones for p in dpdes]
    xticklab = [f"{bk}\n{PDE_LABEL[p]}" for (bk, p) in groups]
    bkcol = {"2D FNO": "#d62728", "1D FNO": "#ff7f0e", "1D DeepONet": "#2ca02c"}
    for ax, (data, title, ylab) in zip(
            axes,
            [(RHOM_SEEDS, r"Gauge-DEPENDENT $\rho_M$ (per seed): unstable", r"$\rho_M$"),
             (RMECH_SEEDS, r"Gauge-INVARIANT $r_{\mathrm{mech}}$ (per seed): stable", r"$r_{\mathrm{mech}}$")]):
        for i, (bk, p) in enumerate(groups):
            pts = data[bk][p]
            ax.scatter([i] * len(pts), pts, s=42, color=bkcol[bk],
                       edgecolor="black", linewidth=0.4, zorder=3)
            ax.plot([i, i], [min(pts), max(pts)], color=bkcol[bk], lw=1.0, alpha=0.6)
        ax.set_xticks(range(len(groups))); ax.set_xticklabels(xticklab, fontsize=7)
        ax.set_ylabel(ylab); ax.set_title(title, fontsize=10); ax.grid(axis="y", ls=":", alpha=0.5)
    axes[0].annotate("same flow,\n$\\rho_M$ flips 0$\\leftrightarrow$1", xy=(3, 0.5), xytext=(3.4, 0.5),
                     fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.7))
    fig.suptitle("Gauge freedom: channel attribution scatters across seeds while physical dissipation does not", y=1.04, fontsize=10)
    _save(fig, "fig_gauge_variance")


def fig_rmech_ordering():
    bkcol = {"2D FNO": "#d62728", "1D FNO": "#ff7f0e", "1D DeepONet": "#2ca02c"}
    mk = {"heat": "o", "advection": "^", "burgers": "s"}
    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    for bk in RMECH:
        for p in PDES:
            rm, _, pit = RMECH[bk][p]
            ax.scatter(pit, max(rm, 1e-4), s=70, marker=mk[p], color=bkcol[bk],
                       edgecolor="black", linewidth=0.5, zorder=3)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"ground-truth dissipation $\Pi^\star$ (log)")
    ax.set_ylabel(r"model $r_{\mathrm{mech}}$ (log)")
    ax.set_title(r"$r_{\mathrm{mech}}$ recovers the ground-truth dissipation ordering")
    # legends
    from matplotlib.lines import Line2D
    leg1 = [Line2D([], [], marker=mk[p], color="w", markerfacecolor="gray",
                   markeredgecolor="black", ms=8, label=PDE_LABEL[p]) for p in PDES]
    leg2 = [Line2D([], [], marker="o", color="w", markerfacecolor=bkcol[bk],
                   markeredgecolor="black", ms=8, label=bk) for bk in RMECH]
    l1 = ax.legend(handles=leg1, title="PDE", fontsize=8, loc="upper left")
    ax.add_artist(l1); ax.legend(handles=leg2, title="backbone", fontsize=8, loc="lower right")
    ax.grid(ls=":", alpha=0.5)
    _save(fig, "fig_rmech_ordering")


def fig_euler_rk4():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    ax = axes[0]
    ax.plot(ERK_RES, EULER_RK4["FNO"]["L2"], "-o", ms=5, color=C["FNO"], label="FNO (baseline)")
    ax.plot(ERK_RES, EULER_RK4["Euler"]["L2"], "-s", ms=5, color="#d62728", label="GENERIC (Euler)")
    ax.plot(ERK_RES, EULER_RK4["RK4"]["L2"], "--^", ms=5, color="#1f77b4", label="GENERIC (RK4+AA)")
    ax.axvline(64, color="black", ls=":", lw=0.8); ax.text(64, ax.get_ylim()[1]*0.9, " train grid", fontsize=8)
    ax.set_xlabel("evaluation resolution"); ax.set_ylabel(r"advection rollout $L^2$")
    ax.set_title("RK4 mitigates the coarse-grid blow-up"); ax.set_xticks(ERK_RES)
    ax.legend(fontsize=8); ax.grid(ls=":", alpha=0.5)
    ax = axes[1]
    ax.plot(ERK_RES, EULER_RK4["Euler"]["rE"], "-s", ms=5, color="#d62728", label="Euler ($\\sim$1e-7)")
    ax.plot(ERK_RES, EULER_RK4["RK4"]["rE"], "--^", ms=5, color="#1f77b4", label="RK4 ($\\sim$1e-3)")
    ax.set_yscale("log"); ax.axvline(64, color="black", ls=":", lw=0.8)
    ax.set_xlabel("evaluation resolution"); ax.set_ylabel(r"energy residual $r_E$ (log)")
    ax.set_title("...but RK4 forfeits exact discrete degeneracy"); ax.set_xticks(ERK_RES)
    ax.legend(fontsize=8); ax.grid(ls=":", alpha=0.5)
    fig.suptitle("Why we keep explicit Euler: the machine-precision discrete guarantee", y=1.04, fontsize=10)
    _save(fig, "fig_euler_rk4")


# ----------------------------------------------------------------------------
# INTERPRETABILITY (LIVE) -- needs torch + model module + a GENERIC checkpoint
# ----------------------------------------------------------------------------
def fig_interpretability(n_steps=12):
    """Roll a trained GENERIC-FNO2d out and plot learned E (flat), learned S
    (rising), and mechanical energy Q=0.5||u||^2 along the trajectory, per PDE.
    Skipped gracefully if torch / the model module / checkpoints are missing."""
    try:
        import torch  # noqa
        import importlib.util, glob
        spec = importlib.util.spec_from_file_location(
            "g2d", os.path.join(os.path.dirname(__file__) or ".", "generic_fno_2d_v2.py"))
        g2d = importlib.util.module_from_spec(spec); spec.loader.exec_module(g2d)
    except Exception as e:
        print(f"  [skip] fig_interpretability (no torch/model module): {e}")
        return
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    pde_ckpt = {p: os.path.join(BASE, f"generic_fno_2d_{p}_nx128.pt") for p in PDES}
    panels = {}
    for p in PDES:
        ck = pde_ckpt[p]
        if not os.path.exists(ck):
            print(f"  [skip] checkpoint not found: {ck}")
            continue
        try:
            model = g2d.GENERIC_FNO2d(degeneracy_construction=True).to(dev)
            state = torch.load(ck, map_location=dev)
            model.load_state_dict(state.get("model", state) if isinstance(state, dict) else state)
            model.eval()
            # band-limited IC consistent with the generators in the module
            u = g2d.make_initial_conditions(p, n=1, nx=128).to(dev) if hasattr(g2d, "make_initial_conditions") \
                else torch.randn(1, 1, 128, 128, device=dev)
            E, S, Q = [], [], []
            with torch.no_grad():
                for _ in range(n_steps):
                    E.append(float(model.E_net(u).mean()))
                    S.append(float(model.S_net(u).mean()))
                    Q.append(float(0.5 * (u ** 2).mean()))
                    u = model(u)
            panels[p] = (np.array(E), np.array(S), np.array(Q))
        except Exception as e:
            print(f"  [skip] {p}: {e}")
    if not panels:
        print("  [skip] fig_interpretability: no panels produced (run on the machine with checkpoints).")
        return
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 3.2), squeeze=False)
    for ax, (p, (E, S, Q)) in zip(axes[0], panels.items()):
        t = np.arange(len(E))
        norm = lambda a: (a - a.min()) / (a.ptp() + 1e-12)
        ax.plot(t, norm(E), "-o", ms=3, color="#1f77b4", label=r"$E[u_t]$ (learned energy)")
        ax.plot(t, norm(S), "-s", ms=3, color="#d62728", label=r"$S[u_t]$ (learned entropy)")
        ax.plot(t, norm(Q), "--", color="black", label=r"$Q=\tfrac12\|u\|^2$")
        ax.set_title(PDE_LABEL[p]); ax.set_xlabel("rollout step"); ax.set_ylabel("normalized")
        ax.legend(fontsize=7); ax.grid(ls=":", alpha=0.5)
    fig.suptitle("Learned thermodynamics along rollouts: $E$ conserved, $S$ tracks $Q$ decay", y=1.04, fontsize=10)
    _save(fig, "fig_interpretability")


# ----------------------------------------------------------------------------
# Optional: reload scalar data from your own pickles instead of the embedded
# verified constants.  Fill in to match your pkl schema, then call in main().
# ----------------------------------------------------------------------------
def load_from_pkls(base=BASE):
    """STUB. Return dicts overriding ACC / RMECH / SUPERRES from the newest
    *.pkl in `base`.  Left unimplemented because the embedded constants are
    already the verified seed-averaged values; wire this up only if you re-run."""
    raise NotImplementedError


# ----------------------------------------------------------------------------
def main():
    print(f"FIGDIR = {FIGDIR}")
    print("Main-text figures:")
    fig_accuracy(); fig_dissipation(); fig_resolution()
    fig_interpretability()
    print("Appendix figures:")
    fig_degeneracy(); fig_gauge_variance(); fig_rmech_ordering(); fig_euler_rk4()
    print("done.")


if __name__ == "__main__":
    main()

FIGDIR = /content/drive/MyDrive/GENERIC_FNO_results/figures
Main-text figures:
  saved /content/drive/MyDrive/GENERIC_FNO_results/figures/fig_accuracy.pdf/.png
  saved /content/drive/MyDrive/GENERIC_FNO_results/figures/fig_dissipation.pdf/.png
  saved /content/drive/MyDrive/GENERIC_FNO_results/figures/fig_resolution.pdf/.png
  [skip] fig_interpretability (no torch/model module): name '__file__' is not defined
Appendix figures:
  saved /content/drive/MyDrive/GENERIC_FNO_results/figures/fig_degeneracy.pdf/.png
  saved /content/drive/MyDrive/GENERIC_FNO_results/figures/fig_gauge_variance.pdf/.png
  saved /content/drive/MyDrive/GENERIC_FNO_results/figures/fig_rmech_ordering.pdf/.png
  saved /content/drive/MyDrive/GENERIC_FNO_results/figures/fig_euler_rk4.pdf/.png
done.
